- _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_itemchild
- _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_other
- _exponent._bronze_allscripts_tw_works_vw.dbo_charge_modifier
- _exponent._bronze_allscripts_tw_works_vw.dbo_cpt4_modifier_de
- _exponent._bronze_allscripts_tw_works_vw.dbo_vendor_item

### Need
- _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_item
- _exponent._bronze_allscripts_tw_works_vw.dbo_order_source_de
- _exponent._bronze_allscripts_tw_works_vw.dbo_idx_user
- _exponent._bronze_allscripts_tw_works_vw.dbo_source_de
- _exponent._bronze_allscripts_tw_works_vw.dbo_activity_type_de

### Maybe Need
- _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity
- _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header
- _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_item

In [0]:

source = 'allscripts_tw'

### Charge Based Procedure Occurrences


### To Do:
 - Join to visit_occurrence and visit_detail table on visitid

In [0]:
WITH charge_procedures AS (
  SELECT
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.id AS STRING)                                  AS source_record_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.encounterid AS STRING)                         AS source_encounter_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.visitid AS STRING)                             AS source_visit_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.billingproviderid AS STRING)                   AS source_provider_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.otherproviderid AS STRING)                     AS source_performing_provider_id,
    DATE(COALESCE(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.postdttm,
                  _exponent._bronze_allscripts_tw_works_vw.dbo_charge.starttime,
                  _exponent._bronze_allscripts_tw_works_vw.dbo_charge.endtime))                              AS procedure_date,
    COALESCE(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.postdttm,
             _exponent._bronze_allscripts_tw_works_vw.dbo_charge.starttime,
             _exponent._bronze_allscripts_tw_works_vw.dbo_charge.endtime)                                    AS procedure_datetime,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge.unitstobillfor AS DOUBLE)                      AS quantity,
    NULLIF(
      REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''),
      ''
    )                                                                                                       AS procedure_source_value,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.entryname AS STRING)                   AS procedure_source_name
  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge
  JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de
    ON _exponent._bronze_allscripts_tw_works_vw.dbo_charge.chargecodede =
       _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.id
  WHERE 1 = 1
    -- keep non-visit "procedure" charges
    AND _exponent._bronze_allscripts_tw_works_vw.dbo_charge.islevelofservicechargeflag = 'N'
    -- keep active-ish billing statuses (drop canceled/removed)
    AND _exponent._bronze_allscripts_tw_works_vw.dbo_charge.billingstatus NOT IN ('C','R')
    -- require CPT present on the charge code dictionary row
    AND NULLIF(
          REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''),
          ''
        ) IS NOT NULL
),

order_item_result_procedures AS (
  SELECT
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header.id AS STRING)                   AS source_record_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header.encounterid AS STRING)          AS source_encounter_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header.visitid AS STRING)              AS source_visit_id,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_order_activity.orderingproviderid AS STRING)          AS source_provider_id,
    DATE(_exponent._bronze_allscripts_tw_works_vw.dbo_item_result.performeddttm)                            AS procedure_date,
    _exponent._bronze_allscripts_tw_works_vw.dbo_item_result.performeddttm                                  AS procedure_datetime,
    CAST(1 AS DOUBLE)                                                                                       AS quantity,
    COALESCE(
      NULLIF(REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.cpt4code,   '[\\s\\u00A0]+', ''), ''),
      NULLIF(REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
    )                                                                                                       AS procedure_source_value,
    CAST(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.entryname AS STRING)             AS procedure_source_name
  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header
  JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity
    ON _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity.orderactivityid =
       _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header.currentorderactivityid
  JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_item_result
    ON _exponent._bronze_allscripts_tw_works_vw.dbo_item_result.orderitemext =
       _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity.ordernumberext
   AND _exponent._bronze_allscripts_tw_works_vw.dbo_item_result.patientid =
       _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header.patientid
  JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de
    ON _exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.id =
       _exponent._bronze_allscripts_tw_works_vw.dbo_item_result.qoclassificationde
  WHERE 1 = 1
    AND _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header.activitytype = 'Order'
    AND _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity.orderstatusde IN (3, 4, 18, 20)
    AND TRIM(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.itemtype) = 'OT'
    AND COALESCE(
          TRIM(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.ordertype),
          TRIM(_exponent._bronze_allscripts_tw_works_vw.dbo_order_activity.ordertype)
        ) <> 'L'
    AND COALESCE(
          NULLIF(REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.cpt4code,   '[\\s\\u00A0]+', ''), ''),
          NULLIF(REGEXP_REPLACE(_exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
        ) IS NOT NULL
),

all_procedure_candidates AS (
  SELECT * FROM order_item_result_procedures
  UNION ALL
  SELECT * FROM charge_procedures
),

procedure_occurrence_shaped AS (
  SELECT
    /* you can replace this with your standard silver surrogate key logic */
    CAST(NULL AS BIGINT)                                                                                   AS procedure_occurrence_id,

    /* TODO: join to your person crosswalk to populate person_id */
    CAST(NULL AS BIGINT)                                                                                   AS person_id,

    /* TODO: map procedure_source_value (CPT/HCPCS) -> standard OMOP concept_id */
    CAST(0 AS INT)                                                                                          AS procedure_concept_id,

    all_procedure_candidates.procedure_date                                                                  AS procedure_date,
    all_procedure_candidates.procedure_datetime                                                              AS procedure_datetime,

    /* common choices: EHR order (38000275), EHR billing record (often local). set per-source if you prefer */
    CAST(38000275 AS INT)                                                                                    AS procedure_type_concept_id,

    CAST(NULL AS INT)                                                                                        AS modifier_concept_id,
    CAST(all_procedure_candidates.quantity AS DOUBLE)                                                        AS quantity,

    /* TODO: join to provider crosswalk if you want provider_id */
    CAST(NULL AS BIGINT)                                                                                     AS provider_id,

    /* TODO: join to visit_occurrence via visit/encounter crosswalk */
    CAST(NULL AS BIGINT)                                                                                     AS visit_occurrence_id,
    CAST(NULL AS BIGINT)                                                                                     AS visit_detail_id,

    CAST(all_procedure_candidates.procedure_source_value AS STRING)                                          AS procedure_source_value,

    /* TODO: map CPT/HCPCS source concept if you maintain source_concept_id */
    CAST(0 AS INT)                                                                                           AS procedure_source_concept_id,

    CAST(NULL AS STRING)                                                                                     AS modifier_source_value,

    /* lineage fields you may want to keep in silver */
    CAST(all_procedure_candidates.source_record_id AS STRING)                                                AS source_record_id,
    CAST(all_procedure_candidates.source_visit_id AS STRING)                                                 AS source_visit_id,
    CAST(all_procedure_candidates.source_encounter_id AS STRING)                                             AS source_encounter_id,
    CAST(all_procedure_candidates.source_provider_id AS STRING)                                              AS source_provider_id,
    CAST(all_procedure_candidates.source_performing_provider_id AS STRING)                                   AS source_performing_provider_id,
    CAST(all_procedure_candidates.procedure_source_name AS STRING)                                           AS procedure_source_name
  FROM all_procedure_candidates
)

SELECT
  procedure_occurrence_id,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_record_id,
  source_visit_id,
  source_encounter_id,
  source_provider_id,
  source_performing_provider_id,
  procedure_source_name
FROM procedure_occurrence_shaped;


In [0]:
%sql
SELECT DISTINCT
 CONCAT('allscripts_tw | ', CAST(dbo_encounter.id AS BIGINT)) AS procedure_occurrence_id,

  -- CONCAT('allscripts_tw | ', dbo_encounter.patientid) AS person_id,
  source_to_person.person_id AS person_id,
  -- COALESCE(source_to_person.person_id, CONCAT('allscripts_tw | '
  COALESCE(standard_concept.concept_id, 0) AS procedure_concept_id,

  CAST(COALESCE(dbo_charge.postdttm, dbo_charge.starttime, dbo_charge.endtime) AS DATE)      AS procedure_date,
  CAST(COALESCE(dbo_charge.postdttm, dbo_charge.starttime, dbo_charge.endtime) AS TIMESTAMP) AS procedure_datetime,

  CAST(44814649 AS INT) AS procedure_type_concept_id,
  CAST(0 AS INT) AS modifier_concept_id,
  CAST(COALESCE(dbo_charge.unitstobillfor, 1) AS DOUBLE) AS quantity,

  -- CAST(COALESCE(dbo_charge.billingproviderid, dbo_charge.otherproviderid) AS BIGINT) AS provider_id,
  source_to_provider.provider_id AS provider_id,
  CAST(dbo_charge.visitid AS BIGINT) AS visit_occurrence_id,
  CAST(NULL AS BIGINT) AS visit_detail_id,

  NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), '') AS procedure_source_value,
  COALESCE(concept.concept_id, 0) AS procedure_source_concept_id,

  primary_modifier.modifier_source_value AS modifier_source_value,
  'allscripts_tw' AS source_system

FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de
  ON dbo_charge_code_de.id = dbo_charge.chargecodede

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_other
  ON dbo_encounter_other.EncounterId = dbo_charge.encounterid

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
  ON dbo_encounter.id = dbo_encounter_other.EncounterId

LEFT JOIN (
  SELECT
    cm.ChargeID,
    NULLIF(REGEXP_REPLACE(CAST(cmd.entrycode AS STRING), '[\\s\\u00A0]+', ''), '') AS modifier_source_value,
    ROW_NUMBER() OVER (PARTITION BY cm.ChargeID ORDER BY cm.ModifierNumber ASC) AS rn
  FROM _exponent._bronze_allscripts_tw_works.dbo_charge_modifier cm
  INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_cpt4_modifier_de cmd
    ON cmd.id = cm.BillingChargeModifierDE
) primary_modifier
  ON primary_modifier.ChargeID = dbo_charge.id
 AND primary_modifier.rn = 1

LEFT JOIN _exponent.omop.concept
  ON concept.concept_code = NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), '')
 AND concept.vocabulary_id IN ('CPT4','HCPCS')
 AND concept.domain_id = 'Procedure'
 AND concept.invalid_reason IS NULL

LEFT JOIN _exponent.omop.concept_relationship
  ON concept_relationship.concept_id_1 = concept.concept_id
 AND concept_relationship.relationship_id = 'Maps to'

LEFT JOIN _exponent.omop.concept standard_concept
  ON standard_concept.concept_id = concept_relationship.concept_id_2
 AND standard_concept.standard_concept = 'S'
 AND standard_concept.invalid_reason IS NULL
 AND standard_concept.domain_id = 'Procedure'
LEFT JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value = CONCAT('allscripts_tw | ', CAST(dbo_encounter.patientid AS BIGINT))
LEFT JOIN _exponent.omop_mapping.source_to_provider
  ON source_to_provider.provider_source_value = CONCAT('allscripts_tw | ', CAST(COALESCE(dbo_charge.billingproviderid, dbo_charge.otherproviderid) AS BIGINT))

WHERE 1 = 1
  AND dbo_charge.islevelofservicechargeflag = 'N'
  AND dbo_charge_code_de.IsLevelOfServiceFlag = 'N'
  AND NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), '') IS NOT NULL
  -- AND dbo_charge.etl_load_ts BETWEEN  CURRENT_DATE() - INTERVAL 14 DAY AND CURRENT_DATE();


### Activity Based Procedure Occurrences


### To Do:
 - Join to visit_occurrence and visit_detail table on visitid

In [0]:
%sql
SELECT DISTINCT
  CONCAT(
    'allscripts_tw|dbo_item_result|',
    CAST(dbo_item_result.id AS STRING)
  ) AS procedure_occurrence_source_value,

  COALESCE(source_to_person.person_id, 0) AS person_id,
  COALESCE(standard_concept.concept_id, 0) AS procedure_concept_id,

  CAST(COALESCE(dbo_item_result.performeddttm, dbo_order_activity_header.createdttm) AS DATE) AS procedure_date,
  CAST(COALESCE(dbo_item_result.performeddttm, dbo_order_activity_header.createdttm) AS TIMESTAMP) AS procedure_datetime,

  CAST(32817 AS INT) AS procedure_type_concept_id, -- hardcoding "EHR order"
  CAST(0 AS INT) AS modifier_concept_id,
  CAST(1 AS DOUBLE) AS quantity,

  source_to_provider.provider_id AS provider_id,

  CAST(dbo_encounter.visitid AS BIGINT) AS visit_occurrence_id,
  CAST(dbo_encounter.visitid AS BIGINT) AS visit_detail_id,

  COALESCE(
    NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code,  '[\\s\\u00A0]+', ''), ''),
    NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
  ) AS procedure_source_value,

  COALESCE(concept.concept_id, 0) AS procedure_source_concept_id,

  NULLIF(REGEXP_REPLACE(dbo_qo_mod_de.entrycode, '[\\s\\u00A0]+', ''), '') AS modifier_source_value,
  'allscripts_tw' AS source_system

FROM _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header
INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity
  ON dbo_order_activity.orderactivityheaderid = dbo_order_activity_header.id

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
  ON dbo_encounter.id = dbo_order_activity_header.encounterid

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_item_result
  ON dbo_item_result.orderitemext = dbo_order_activity.ordernumberext
 AND dbo_item_result.patientid   = dbo_encounter.patientid

INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de
  ON dbo_qo_classification_de.id = dbo_item_result.qoclassificationde

LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_mod_de
  ON dbo_qo_mod_de.id = COALESCE(dbo_item_result.qomod1de, dbo_item_result.qomod2de, dbo_item_result.qomod3de)

LEFT OUTER JOIN _exponent.omop.concept
  ON concept.concept_code = COALESCE(
        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code,  '[\\s\\u00A0]+', ''), ''),
        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
      )
 AND concept.vocabulary_id IN ('CPT4','HCPCS')
 AND concept.domain_id = 'Procedure'
 AND concept.invalid_reason IS NULL

LEFT JOIN _exponent.omop.concept_relationship
  ON concept_relationship.concept_id_1 = concept.concept_id
 AND concept_relationship.relationship_id = 'Maps to'

LEFT JOIN _exponent.omop.concept standard_concept
  ON standard_concept.concept_id = concept_relationship.concept_id_2
 AND standard_concept.standard_concept = 'S'
 AND standard_concept.invalid_reason IS NULL
 AND standard_concept.domain_id = 'Procedure'
LEFT JOIN _exponent.omop_mapping.source_to_person
  ON source_to_person.person_source_value = CONCAT('allscripts_tw | ', CAST(dbo_encounter.patientid AS BIGINT))
LEFT JOIN _exponent.omop_mapping.source_to_provider
  ON source_to_provider.provider_source_value = CONCAT('allscripts_tw | ', CAST(dbo_order_activity.orderingproviderid AS BIGINT))

WHERE 1 = 1
  AND dbo_order_activity_header.activitytype = 'Order'
  AND dbo_order_activity.orderstatusde IN (3, 4, 18, 20)
  AND dbo_qo_classification_de.itemtype = 'OT'
  AND dbo_qo_classification_de.ordertype <> 'L'
  AND COALESCE(
        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code,  '[\\s\\u00A0]+', ''), ''),
        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
      ) IS NOT NULL;


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW procedure_occurrence_silver AS
WITH procedure_occurrence_charge AS (
  SELECT DISTINCT

    CONCAT('allscripts_tw|dbo_charge|', CAST(dbo_charge.id AS STRING)) AS procedure_occurrence_source_value,

    source_to_person.person_id AS person_id,
    COALESCE(standard_concept.concept_id, 0) AS procedure_concept_id,

    CAST(COALESCE(dbo_charge.postdttm, dbo_charge.starttime, dbo_charge.endtime) AS DATE)      AS procedure_date,
    CAST(COALESCE(dbo_charge.postdttm, dbo_charge.starttime, dbo_charge.endtime) AS TIMESTAMP) AS procedure_datetime,

    CAST(44814649 AS INT) AS procedure_type_concept_id,
    CAST(0 AS INT) AS modifier_concept_id,
    CAST(COALESCE(dbo_charge.unitstobillfor, 1) AS DOUBLE) AS quantity,

    source_to_provider.provider_id AS provider_id,
    CAST(dbo_charge.visitid AS BIGINT) AS visit_occurrence_id,
    CAST(NULL AS BIGINT) AS visit_detail_id,

    NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), '') AS procedure_source_value,
    COALESCE(concept.concept_id, 0) AS procedure_source_concept_id,

    primary_modifier.modifier_source_value AS modifier_source_value,
    'allscripts_tw' AS source_system,

    CONCAT_WS(
      '|',
      CAST(COALESCE(source_to_person.person_id, 0) AS STRING),
      CAST(COALESCE(standard_concept.concept_id, 0) AS STRING),
      CAST(CAST(COALESCE(dbo_charge.postdttm, dbo_charge.starttime, dbo_charge.endtime) AS TIMESTAMP) AS STRING),
      CAST(COALESCE(source_to_provider.provider_id, 0) AS STRING),
      CAST(COALESCE(dbo_charge.visitid, 0) AS STRING),
      COALESCE(NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), ''), '')
    ) AS dedupe_key,

    -- give charge a lower priority than orders (orders win ties)
    CAST(2 AS INT) AS source_priority

  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_charge

  INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_charge_code_de
    ON dbo_charge_code_de.id = dbo_charge.chargecodede

  INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter_other
    ON dbo_encounter_other.EncounterId = dbo_charge.encounterid

  INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
    ON dbo_encounter.id = dbo_encounter_other.EncounterId

  LEFT JOIN (
    SELECT
      cm.ChargeID,
      NULLIF(REGEXP_REPLACE(CAST(cmd.entrycode AS STRING), '[\\s\\u00A0]+', ''), '') AS modifier_source_value,
      ROW_NUMBER() OVER (PARTITION BY cm.ChargeID ORDER BY cm.ModifierNumber ASC) AS rn
    FROM _exponent._bronze_allscripts_tw_works.dbo_charge_modifier cm
    INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_cpt4_modifier_de cmd
      ON cmd.id = cm.BillingChargeModifierDE
  ) primary_modifier
    ON primary_modifier.ChargeID = dbo_charge.id
   AND primary_modifier.rn = 1

  LEFT JOIN _exponent.omop.concept
    ON concept.concept_code = NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), '')
   AND concept.vocabulary_id IN ('CPT4','HCPCS')
   AND concept.domain_id = 'Procedure'
   AND concept.invalid_reason IS NULL

  LEFT JOIN _exponent.omop.concept_relationship
    ON concept_relationship.concept_id_1 = concept.concept_id
   AND concept_relationship.relationship_id = 'Maps to'

  LEFT JOIN _exponent.omop.concept standard_concept
    ON standard_concept.concept_id = concept_relationship.concept_id_2
   AND standard_concept.standard_concept = 'S'
   AND standard_concept.invalid_reason IS NULL
   AND standard_concept.domain_id = 'Procedure'

  LEFT JOIN _exponent.omop_mapping.source_to_person
    ON source_to_person.person_source_value = CONCAT('allscripts_tw | ', CAST(dbo_encounter.patientid AS BIGINT))

  LEFT JOIN _exponent.omop_mapping.source_to_provider
    ON source_to_provider.provider_source_value = CONCAT(
      'allscripts_tw | ',
      CAST(COALESCE(dbo_charge.billingproviderid, dbo_charge.otherproviderid) AS BIGINT)
    )

  WHERE 1 = 1
    AND dbo_charge.islevelofservicechargeflag = 'N'
    AND dbo_charge_code_de.IsLevelOfServiceFlag = 'N'
    AND NULLIF(REGEXP_REPLACE(dbo_charge_code_de.cpt4code, '[\\s\\u00A0]+', ''), '') IS NOT NULL
),
procedure_occurrence_order AS (
  SELECT DISTINCT
    CONCAT('allscripts_tw|dbo_item_result|', CAST(dbo_item_result.id AS STRING)) AS procedure_occurrence_source_value,

    COALESCE(source_to_person.person_id, 0) AS person_id,
    COALESCE(standard_concept.concept_id, 0) AS procedure_concept_id,

    CAST(COALESCE(dbo_item_result.performeddttm, dbo_order_activity_header.createdttm) AS DATE)      AS procedure_date,
    CAST(COALESCE(dbo_item_result.performeddttm, dbo_order_activity_header.createdttm) AS TIMESTAMP) AS procedure_datetime,

    CAST(32817 AS INT) AS procedure_type_concept_id,
    CAST(0 AS INT) AS modifier_concept_id,
    CAST(1 AS DOUBLE) AS quantity,

    source_to_provider.provider_id AS provider_id,

    CAST(dbo_encounter.visitid AS BIGINT) AS visit_occurrence_id,
    CAST(dbo_encounter.visitid AS BIGINT) AS visit_detail_id,

    COALESCE(
      NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code,  '[\\s\\u00A0]+', ''), ''),
      NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
    ) AS procedure_source_value,

    COALESCE(concept.concept_id, 0) AS procedure_source_concept_id,

    NULLIF(REGEXP_REPLACE(dbo_qo_mod_de.entrycode, '[\\s\\u00A0]+', ''), '') AS modifier_source_value,
    'allscripts_tw' AS source_system,

    -- dedupe_key — includes CPT/HCPCS + datetime + visit + person + provider
    CONCAT_WS(
      '|',
      CAST(COALESCE(source_to_person.person_id, 0) AS STRING),
      CAST(COALESCE(standard_concept.concept_id, 0) AS STRING),
      CAST(CAST(COALESCE(dbo_item_result.performeddttm, dbo_order_activity_header.createdttm) AS TIMESTAMP) AS STRING),
      CAST(COALESCE(source_to_provider.provider_id, 0) AS STRING),
      CAST(COALESCE(dbo_encounter.visitid, 0) AS STRING),
      COALESCE(
        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code,  '[\\s\\u00A0]+', ''), ''),
        NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
      )
    ) AS dedupe_key,

    -- orders should win over charges when duplicates exist
    CAST(1 AS INT) AS source_priority

  FROM _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity_header

  INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_order_activity
    ON dbo_order_activity.orderactivityheaderid = dbo_order_activity_header.id

  INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_encounter
    ON dbo_encounter.id = dbo_order_activity_header.encounterid

  INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_item_result
    ON dbo_item_result.orderitemext = dbo_order_activity.ordernumberext
   AND dbo_item_result.patientid   = dbo_encounter.patientid

  INNER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_classification_de
    ON dbo_qo_classification_de.id = dbo_item_result.qoclassificationde

  LEFT OUTER JOIN _exponent._bronze_allscripts_tw_works_vw.dbo_qo_mod_de
    ON dbo_qo_mod_de.id = COALESCE(dbo_item_result.qomod1de, dbo_item_result.qomod2de, dbo_item_result.qomod3de)

  LEFT OUTER JOIN _exponent.omop.concept
    ON concept.concept_code = COALESCE(
          NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code,  '[\\s\\u00A0]+', ''), ''),
          NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
        )
   AND concept.vocabulary_id IN ('CPT4','HCPCS')
   AND concept.domain_id = 'Procedure'
   AND concept.invalid_reason IS NULL

  LEFT JOIN _exponent.omop.concept_relationship
    ON concept_relationship.concept_id_1 = concept.concept_id
   AND concept_relationship.relationship_id = 'Maps to'

  LEFT JOIN _exponent.omop.concept standard_concept
    ON standard_concept.concept_id = concept_relationship.concept_id_2
   AND standard_concept.standard_concept = 'S'
   AND standard_concept.invalid_reason IS NULL
   AND standard_concept.domain_id = 'Procedure'

  LEFT JOIN _exponent.omop_mapping.source_to_person
    ON source_to_person.person_source_value = CONCAT('allscripts_tw | ', CAST(dbo_encounter.patientid AS BIGINT))

  LEFT JOIN _exponent.omop_mapping.source_to_provider
    ON source_to_provider.provider_source_value = CONCAT('allscripts_tw | ', CAST(dbo_order_activity.orderingproviderid AS BIGINT))

  WHERE 1 = 1
    AND dbo_order_activity_header.activitytype = 'Order'
    AND dbo_order_activity.orderstatusde IN (3, 4, 18, 20)
    AND dbo_qo_classification_de.itemtype = 'OT'
    AND dbo_qo_classification_de.ordertype <> 'L'
    AND COALESCE(
          NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.cpt4code,  '[\\s\\u00A0]+', ''), ''),
          NULLIF(REGEXP_REPLACE(dbo_qo_classification_de.hicpcscode,'[\\s\\u00A0]+', ''), '')
        ) IS NOT NULL
),
unioned AS (
  SELECT * FROM procedure_occurrence_order
  UNION ALL
  SELECT * FROM procedure_occurrence_charge
),
deduped AS (
  SELECT
    unioned.*,
    ROW_NUMBER() OVER (
      PARTITION BY dedupe_key
      ORDER BY source_priority ASC, procedure_datetime ASC
    ) AS rn
  FROM unioned
)
SELECT
  procedure_occurrence_source_value,
  person_id,
  procedure_concept_id,
  procedure_date,
  procedure_datetime,
  procedure_type_concept_id,
  modifier_concept_id,
  quantity,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  procedure_source_value,
  procedure_source_concept_id,
  modifier_source_value,
  source_system
FROM deduped
WHERE rn = 1;


In [0]:
%sql
SELECT * FROM procedure_occurrence_silver LIMIT 10